In [ ]:
# -------------------------------------------------------
# Imports
# Purpose: Load required libraries — TypedDict for typed state schema,
# StateGraph/START/END for graph construction, ChatPromptTemplate for
# structured prompt building, and ChatBedrockConverse for AWS Bedrock LLM access.
# -------------------------------------------------------
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_aws import ChatBedrockConverse

In [ ]:
# -------------------------------------------------------
# LLM Initialization
# Purpose: Instantiate the Claude 3.5 Sonnet model via AWS Bedrock.
# This single LLM instance is shared across all nodes to avoid
# redundant client creation overhead.
# -------------------------------------------------------
llm = ChatBedrockConverse(
    model="anthropic.claude-3-5-sonnet-20241022-v2:0",
    region_name="us-east-1",
)

In [ ]:
# -------------------------------------------------------
# Valid Routes Registry
# Purpose: Whitelist of allowed routing targets. Used to validate
# LLM output and prevent silent misrouting on hallucinated values.
# -------------------------------------------------------
VALID_ROUTES = {"medical", "news", "hospital"}

In [ ]:
# -------------------------------------------------------
# Shared State Schema
# Purpose: Defines the data contract flowing through the entire graph.
# - question : raw user input passed to supervisor and all agents
# - route    : supervisor's routing decision (controls conditional edge)
# - answer   : final response produced by the selected agent
# -------------------------------------------------------
class AgentState(TypedDict):
    question: str
    route: str
    answer: str

In [ ]:
# -------------------------------------------------------
# Supervisor Prompt Template
# Purpose: Instructs the LLM to act as a router — maps user intent
# to exactly one of the three available agents. Strict output format
# (single word) keeps parsing simple and deterministic.
# -------------------------------------------------------
supervisor_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a supervisor. Route to exactly one agent:
- medical  → disease, treatment, drugs, diagnosis
- news     → current events, headlines, updates
- hospital → finding hospitals, clinics, appointments
Return ONLY one word: medical | news | hospital"""),
    ("human", "{question}"),
])

In [ ]:
# -------------------------------------------------------
# Supervisor Chain
# Purpose: Composes the prompt template with the LLM using LangChain's
# pipe operator (LCEL). Invoking this chain returns the LLM's routing decision.
# -------------------------------------------------------
supervisor_chain = supervisor_prompt | llm

In [ ]:
# -------------------------------------------------------
# Supervisor Node
# Purpose: Entry node of the graph. Invokes the LLM to classify the
# user's question and determine which specialist agent should handle it.
# Falls back to "medical" if LLM returns an unrecognized value.
# -------------------------------------------------------
def supervisor(state: AgentState) -> dict:
    raw = supervisor_chain.invoke({"question": state["question"]}).content.strip().lower()

    # Validate LLM output against whitelist; default to "medical" on mismatch
    route = next((r for r in VALID_ROUTES if r in raw), "medical")
    print(f"Supervisor → {route}")
    return {"route": route}

In [ ]:
# -------------------------------------------------------
# Medical Agent Node
# Purpose: Handles questions about diseases, treatments, and drugs.
# In production, replace the hardcoded string with a real LLM chain
# or RAG pipeline over medical literature (e.g., PubMed).
# -------------------------------------------------------
def medical_agent(state: AgentState) -> dict:
    print("Medical Agent Running")
    return {"answer": "Medical Agent: Diabetes treatment includes insulin and lifestyle changes."}


In [ ]:
# -------------------------------------------------------
# News Agent Node
# Purpose: Handles questions about current events and health news.
# In production, wire to a news API (e.g., NewsAPI, web search tool)
# to fetch live, up-to-date information.
# -------------------------------------------------------
def news_agent(state: AgentState) -> dict:
    print("News Agent Running")
    return {"answer": "News Agent: WHO released new diabetes guidelines."}

In [ ]:
# -------------------------------------------------------
# Hospital Agent Node
# Purpose: Handles queries about nearby hospitals, clinics, or appointments.
# In production, integrate with a geolocation API or hospital directory
# using the user's location context.
# -------------------------------------------------------
def hospital_agent(state: AgentState) -> dict:
    print("Hospital Agent Running")
    return {"answer": "Hospital Agent: Apollo Hospital is nearby."}

In [ ]:
# -------------------------------------------------------
# Graph Construction
# Purpose: Assembles the multi-agent workflow as a directed graph.
# Nodes represent processing units; edges define execution flow.
# -------------------------------------------------------
builder = StateGraph(AgentState)

In [ ]:
# Register all nodes (supervisor + 3 specialist agents)
builder.add_node("supervisor", supervisor)
builder.add_node("medical", medical_agent)
builder.add_node("news", news_agent)
builder.add_node("hospital", hospital_agent)


In [ ]:
# -------------------------------------------------------
# Edge Wiring
# Purpose: Defines control flow between nodes.
# - START → supervisor     : graph always begins at the supervisor
# - supervisor → [agents]  : conditional routing based on state["route"]
# - [agents] → END         : each agent terminates the graph after responding
# -------------------------------------------------------
builder.add_edge(START, "supervisor")

# Conditional edge reads state["route"] to select the next node dynamically
builder.add_conditional_edges(
    "supervisor",
    lambda state: state["route"],  # routing key extractor
    {
        "medical": "medical",
        "news": "news",
        "hospital": "hospital",
    },
)

builder.add_edge("medical", END)
builder.add_edge("news", END)
builder.add_edge("hospital", END)

In [ ]:
# -------------------------------------------------------
# Graph Compilation
# Purpose: Validates the graph structure (checks for orphan nodes,
# missing edges, unreachable END) and returns an executable Pregel runnable.
# -------------------------------------------------------
graph = builder.compile()
graph

In [ ]:
# -------------------------------------------------------
# Graph Invocation
# Purpose: Runs the compiled graph with an initial state.
# Execution flows: START → supervisor → [medical|news|hospital] → END
# -------------------------------------------------------
result = graph.invoke({
    "question": "What is the treatment for diabetes?",
    "route": "",   # populated by supervisor node
    "answer": "",  # populated by selected agent node
})

In [ ]:
# -------------------------------------------------------
# Output
# Purpose: Display the final answer produced by the selected agent.
# -------------------------------------------------------
print(f"\nFinal Answer:\n{result['answer']}")